# Robustness Checks

This notebook evaluates the robustness of the primary econometric result from `Primary_Econometric_Analysis.ipynb`. It runs end-to-end on the clean panel dataset at `Data.clean/panel_fixed_effects_data.csv` and compares the main specification with alternative control sets, sample definitions, functional forms, inference methods, and an identification-relevant placebo test.

## 1. Main result and declaration

The primary analysis is a **causal** exercise. It estimates the within-country effect of female secondary school enrollment on fertility rates using a two-way fixed effects model with country and year fixed effects.

**Main result:** The preferred specification produced a coefficient of approximately `-0.0015` on `Female_Secondary_Enrollment_Rate` with a p-value close to `0.948`. This means that we do not find a statistically significant within-country relationship between female secondary enrollment and fertility in this sample.

In [4]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'Data.clean').exists():
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'Data.clean' / 'panel_fixed_effects_data.csv'
OUTPUT_PATH = ROOT / 'Outputs' / 'tables' / 'robustness_analysis_table.csv'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(DATA_PATH)
panel = panel.dropna(subset=['Female_Secondary_Enrollment_Rate', 'Fertility_Rate']).copy()
panel['Log_Fertility_Rate'] = np.log(panel['Fertility_Rate'])
panel['Future_Enrollment_Rate'] = panel.groupby('Country_Name')['Female_Secondary_Enrollment_Rate'].shift(-1)
panel.head()

,Country_Name,Country Code,Year,Female_Secondary_Enrollment_Rate,Fertility_Rate,Log_Fertility_Rate,Future_Enrollment_Rate
0,Burkina Faso,BFA,2005,11.045490,6.182,1.821642,11.844220
1,Burkina Faso,BFA,2006,11.844220,6.170,1.819699,12.869120
2,Burkina Faso,BFA,2007,12.869120,6.110,1.809927,15.249680
3,Burkina Faso,BFA,2008,15.249680,6.050,1.800058,22.436952
4,Burkina Faso,BFA,2012,22.436952,5.793,1.756650,25.124665


In [5]:
def estimate(formula, data, cluster=True):
    if cluster:
        return smf.ols(formula, data=data).fit(
            cov_type='cluster',
            cov_kwds={'groups': data['Country_Name']},
        )
    return smf.ols(formula, data=data).fit(cov_type='HC1')

specifications = [
    (
        'Main: TWFE (cluster)',
        'Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Country_Name) + C(Year)',
        panel,
        True,
        'Fertility_Rate',
        'Main country and year fixed effects estimate with clustered SEs.',
    ),
    (
        'Country FE only',
        'Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Country_Name)',
        panel,
        True,
        'Fertility_Rate',
        'Alternative control set with country fixed effects only.',
    ),
    (
        'Year FE only',
        'Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Year)',
        panel,
        True,
        'Fertility_Rate',
        'Alternative control set with year fixed effects only.',
    ),
    (
        'Log Fertility (TWFE)',
        'Log_Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Country_Name) + C(Year)',
        panel,
        True,
        'Log_Fertility_Rate',
        'Alternative functional form using the log of fertility.',
    ),
    (
        'No Mali',
        'Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Country_Name) + C(Year)',
        panel[panel['Country_Name'] != 'Mali'],
        True,
        'Fertility_Rate',
        'Alternative sample excluding Mali to test sample sensitivity.',
    ),
    (
        'Future Enrollment Placebo',
        'Fertility_Rate ~ Future_Enrollment_Rate + C(Country_Name) + C(Year)',
        panel.dropna(subset=['Future_Enrollment_Rate']),
        True,
        'Fertility_Rate',
        'Identification-relevant placebo test using next-year enrollment.',
    ),
    (
        'Main: TWFE (HC1)',
        'Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Country_Name) + C(Year)',
        panel,
        False,
        'Fertility_Rate',
        'Alternative inference using heteroskedasticity-robust SEs instead of clustering.',
    ),
]

rows = []
for label, formula, data, cluster, outcome, note in specifications:
    model = estimate(formula, data, cluster=cluster)
    coef_label = (
        'Female_Secondary_Enrollment_Rate'
        if 'Female_Secondary_Enrollment_Rate' in model.params
        else 'Future_Enrollment_Rate'
    )
    rows.append({
        'Specification': label,
        'Outcome': outcome,
        'Coefficient': model.params[coef_label],
        'Std_Error': model.bse[coef_label],
        'p_value': model.pvalues[coef_label],
        'N': int(model.nobs),
        'R_squared': model.rsquared,
        'Note': note,
    })

robustness_long = pd.DataFrame(rows).set_index('Specification')
robustness_long['Coefficient'] = robustness_long['Coefficient'].round(4)
robustness_long['Std_Error'] = robustness_long['Std_Error'].round(4)
robustness_long['p_value'] = robustness_long['p_value'].apply(lambda x: '<0.001' if x < 0.001 else round(x, 3))
robustness_long['R_squared'] = robustness_long['R_squared'].round(3)

robustness_table = robustness_long.T.loc[
    ['Outcome', 'Coefficient', 'Std_Error', 'p_value', 'N', 'R_squared', 'Note']
]
robustness_table.to_csv(OUTPUT_PATH)
robustness_table


Specification,Main: TWFE (cluster),Country FE only,Year FE only,Log Fertility (TWFE),No Mali,Future Enrollment Placebo,Main: TWFE (HC1)
Outcome,Fertility_Rate,Fertility_Rate,Fertility_Rate,Log_Fertility_Rate,Fertility_Rate,Fertility_Rate,Fertility_Rate
Coefficient,-0.0015,-0.0562,-0.0572,-0.0004,0.0124,-0.0024,-0.0015
Std_Error,0.0231,0.0076,0.0185,0.0056,0.0124,0.0138,0.0101
p_value,0.948,<0.001,0.002,0.947,0.314,0.862,0.88
N,70,70,70,70,54,66,70
R_squared,0.951,0.867,0.594,0.939,0.959,0.954,0.951
Note,Main country and year fixed effects estimate w...,Alternative control set with country fixed eff...,Alternative control set with year fixed effect...,Alternative functional form using the log of f...,Alternative sample excluding Mali to test samp...,Identification-relevant placebo test using nex...,Alternative inference using heteroskedasticity...


## 3. Robustness table

The table below presents the main specification in column 1 and the robustness checks in the subsequent columns. Each column reports the sample size `N`, coefficient estimate, standard error, p-value, R², and a short note describing the variation from the main specification.

## 4. Interpretation of robustness checks

- **Alternative control sets (Country FE only / Year FE only):** These checks show whether the result depends critically on the two-way fixed effects design. If the coefficient remains negative and roughly similar, the main finding is more credible. If it changes dramatically, the result is sensitive to omitted common or country-specific trends.
- **Alternative functional form (Log Fertility):** This check tests whether the result is driven by the level scale of fertility. If the sign and significance are similar, the main result is not an artifact of the linear outcome assumption.
- **Alternative sample (No Mali):** Dropping one country tests whether a single country drives the effect. If the estimate remains weak and insignificant, it suggests the main result is not unduly driven by Mali.
- **Alternative inference (HC1):** This check evaluates whether the conclusion depends on clustering. If the coefficient still fails to reach significance under robust standard errors, the inference is not driven only by the clustering decision.
- **Identification placebo (Future Enrollment):** This causal check tests whether the model is capturing spurious correlations or reverse timing. In a credible causal design, future enrollment should not predict current fertility. A significant placebo result would indicate a concern with the identifying assumptions.

## 5. Summary

The robustness analysis presents a set of alternative specifications that meaningfully stress-test the main estimate. The full table is also saved to `Outputs/tables/robustness_analysis_table.csv` for reproducibility.